In [1]:
%load_ext rpy2.ipython

In [7]:
%%R

install.packages(c("DBI", "RSQLite", "readr", "dplyr"), repos = "https://cloud.r-project.org")

library(DBI)
library(RSQLite)
library(readr)
library(dplyr)

# Create SQLite database connection
conn <- dbConnect(SQLite(), ":memory:")

# Write dataframes into SQLite tables
dbWriteTable(conn, "orders", orders, overwrite = TRUE)
dbWriteTable(conn, "deliveries", deliveries, overwrite = TRUE)
dbWriteTable(conn, "complaints", complaints, overwrite = TRUE)
dbWriteTable(conn, "hubs", hubs, overwrite = TRUE)
dbWriteTable(conn, "vehicles", vehicles, overwrite = TRUE)
dbWriteTable(conn, "incidents", incidents, overwrite = TRUE)

print(dbListTables(conn))

[1] "complaints" "deliveries" "hubs"       "incidents"  "orders"    
[6] "vehicles"  


Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cloud.r-project.org/src/contrib/DBI_1.3.0.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/RSQLite_3.52.0.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/readr_2.2.0.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/dplyr_1.2.1.tar.gz'

The downloaded source packages are in
	‘/tmp/Rtmp68Z6U4/downloaded_packages’
In addition: Warning message:
call dbDisconnect() when finished working with a connection 


In [5]:
import pandas as pd
import os
import zipfile
from google.colab import files

# Define the base path for the extracted datasets
base_path = 'northstar_dataset/northstar_dataset'
zip_file_name = 'northstar_dataset.zip'
extracted_base_dir = 'northstar_dataset'

# Ensure the base_path exists by attempting to extract the zip if necessary
if not os.path.exists(base_path):
    print(f"'{base_path}' not found. Attempting to extract '{zip_file_name}'.")

    # If the zip file is not on disk, prompt the user to upload it.
    if not os.path.exists(zip_file_name):
        print(f"'{zip_file_name}' not found. Please ensure it's uploaded using files.upload().")
        uploaded_files = files.upload() # This will be interactive
        if zip_file_name not in uploaded_files:
            raise FileNotFoundError(f"'{zip_file_name}' was not found in the uploaded files. Please ensure you upload the correct zip file.")

        # Save the uploaded file to disk for consistency
        with open(zip_file_name, 'wb') as f:
            f.write(uploaded_files[zip_file_name])
        print(f"'{zip_file_name}' saved from 'uploaded' to disk.")

    # Now that we ensured zip_file_name is on disk, proceed with extraction
    try:
        with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
            zip_ref.extractall(extracted_base_dir)
        print(f"'{zip_file_name}' extracted successfully to '{extracted_base_dir}'.")
    except Exception as e:
        print(f"Error during zip extraction: {e}")
        raise FileNotFoundError(f"Failed to extract '{zip_file_name}'.")

# Load all required CSVs into pandas DataFrames
orders_df = pd.read_csv(os.path.join(base_path, 'orders.csv'))
deliveries_df = pd.read_csv(os.path.join(base_path, 'deliveries.csv'))
complaints_df = pd.read_csv(os.path.join(base_path, 'complaints.csv'))
hubs_df = pd.read_csv(os.path.join(base_path, 'hubs.csv'))
vehicles_df = pd.read_csv(os.path.join(base_path, 'vehicles.csv'))
incidents_df = pd.read_csv(os.path.join(base_path, 'incidents.csv'))

print("All required DataFrames loaded successfully for R environment transfer.")

'northstar_dataset/northstar_dataset' not found. Attempting to extract 'northstar_dataset.zip'.
'northstar_dataset.zip' not found. Please ensure it's uploaded using files.upload().


Saving northstar_dataset.zip to northstar_dataset.zip
'northstar_dataset.zip' saved from 'uploaded' to disk.
'northstar_dataset.zip' extracted successfully to 'northstar_dataset'.
All required DataFrames loaded successfully for R environment transfer.


In [6]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri

# Activate the pandas to R data frame conversion
pandas2ri.activate()

# Transfer pandas DataFrames to R environment
ro.globalenv['orders'] = orders_df
ro.globalenv['deliveries'] = deliveries_df
ro.globalenv['complaints'] = complaints_df
ro.globalenv['hubs'] = hubs_df
ro.globalenv['vehicles'] = vehicles_df
ro.globalenv['incidents'] = incidents_df

print("Pandas DataFrames successfully transferred to R environment.")

Pandas DataFrames successfully transferred to R environment.


Now that the dataframes are transferred to R, we can use them in the R environment.

In [8]:
%%R

query_zone <- "
WITH complaint_summary AS (
  SELECT order_id, COUNT(complaint_id) AS complaint_count
  FROM complaints
  GROUP BY order_id
)

SELECT
  CASE
    WHEN LOWER(o.pickup_zone) = 'ctr' THEN 'Central'
    ELSE UPPER(SUBSTR(o.pickup_zone,1,1)) || LOWER(SUBSTR(o.pickup_zone,2))
  END AS pickup_zone_clean,

  COUNT(d.delivery_id) AS deliveries,

  SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,

  ROUND(
    100.0 * SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END)
    / COUNT(d.delivery_id), 2
  ) AS failure_rate_pct,

  SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed,

  ROUND(
    100.0 * SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END)
    / COUNT(d.delivery_id), 2
  ) AS delay_rate_pct,

  ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_rating,

  ROUND(
    100.0 * SUM(CASE WHEN c.complaint_count > 0 THEN 1 ELSE 0 END)
    / COUNT(d.delivery_id), 2
  ) AS complaint_rate_pct

FROM deliveries d
JOIN orders o
  ON d.order_id = o.order_id

LEFT JOIN complaint_summary c
  ON o.order_id = c.order_id

GROUP BY pickup_zone_clean
ORDER BY failure_rate_pct DESC;
"

zone_results <- dbGetQuery(conn, query_zone)
print(zone_results)

  pickup_zone_clean deliveries failed failure_rate_pct delayed delay_rate_pct
1           Central        174     33            18.97      51          29.31
2             North        135     22            16.30      21          15.56
3         Riverside        119     18            15.13      25          21.01
4              West        114     14            12.28      21          18.42
5              East        156     19            12.18      31          19.87
6           Airport        113     12            10.62      31          27.43
7             South        139     14            10.07      22          15.83
  avg_rating complaint_rate_pct
1       3.55              21.26
2       3.90              25.19
3       3.86              26.05
4       3.90              14.91
5       3.91              22.44
6       3.98              20.35
7       4.05              23.02


In [9]:
%%R

query_hub <- "
WITH complaint_summary AS (
  SELECT order_id, COUNT(complaint_id) AS complaint_count
  FROM complaints
  GROUP BY order_id
)

SELECT
  h.hub_id,
  h.hub_name,
  h.zone,

  COUNT(d.delivery_id) AS deliveries,

  SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,

  ROUND(
    100.0 * SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END)
    / COUNT(d.delivery_id), 2
  ) AS failure_rate_pct,

  SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed,

  ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_rating,

  ROUND(
    100.0 * SUM(CASE WHEN c.complaint_count > 0 THEN 1 ELSE 0 END)
    / COUNT(d.delivery_id), 2
  ) AS complaint_rate_pct

FROM deliveries d
JOIN hubs h
  ON d.hub_id = h.hub_id

JOIN orders o
  ON d.order_id = o.order_id

LEFT JOIN complaint_summary c
  ON o.order_id = c.order_id

GROUP BY h.hub_id, h.hub_name, h.zone
ORDER BY failure_rate_pct DESC;
"

hub_results <- dbGetQuery(conn, query_hub)
print(hub_results)

  hub_id       hub_name      zone deliveries failed failure_rate_pct delayed
1    H08  Midtown Relay   Central        128     26            20.31      22
2    H05   Central Core   Central        115     23            20.00      25
3    H06    Airport Hub   Airport        104     15            14.42      27
4    H04      West Gate      West        127     16            12.60      28
5    H01 North Exchange     North        136     17            12.50      26
6    H07  Riverside Hub Riverside        115     14            12.17      25
7    H02     South Link     South        106     10             9.43      26
8    H03      East Dock      East        119     11             9.24      23
  avg_rating complaint_rate_pct
1       3.88              24.22
2       3.67              24.35
3       3.88              20.19
4       3.92              20.47
5       3.84              19.12
6       3.88              26.09
7       3.95              15.09
8       3.90              26.05


In [10]:
%%R

query_service <- "
WITH complaint_summary AS (
  SELECT order_id, COUNT(complaint_id) AS complaint_count
  FROM complaints
  GROUP BY order_id
)

SELECT
  o.service_type,

  COUNT(d.delivery_id) AS deliveries,

  SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,

  ROUND(
    100.0 * SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END)
    / COUNT(d.delivery_id), 2
  ) AS failure_rate_pct,

  SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed,

  ROUND(
    100.0 * SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END)
    / COUNT(d.delivery_id), 2
  ) AS delay_rate_pct,

  ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_rating,

  ROUND(
    100.0 * SUM(CASE WHEN c.complaint_count > 0 THEN 1 ELSE 0 END)
    / COUNT(d.delivery_id), 2
  ) AS complaint_rate_pct

FROM deliveries d
JOIN orders o
  ON d.order_id = o.order_id

LEFT JOIN complaint_summary c
  ON o.order_id = c.order_id

GROUP BY o.service_type
ORDER BY failure_rate_pct DESC;
"

service_results <- dbGetQuery(conn, query_service)
print(service_results)

  service_type deliveries failed failure_rate_pct delayed delay_rate_pct
1     Business        126     25            19.84      28          22.22
2      Medical        108     16            14.81      22          20.37
3    Passenger        262     38            14.50      53          20.23
4       Retail        224     28            12.50      50          22.32
5       Parcel        230     25            10.87      49          21.30
  avg_rating complaint_rate_pct
1       3.85              22.22
2       3.84              19.44
3       3.85              22.52
4       3.87              23.21
5       3.90              21.30


In [11]:
%%R

query_override <- "
WITH complaint_summary AS (
  SELECT order_id, COUNT(complaint_id) AS complaint_count
  FROM complaints
  GROUP BY order_id
)

SELECT
  CASE
    WHEN d.manual_route_override_count = 0 THEN 'No override'
    WHEN d.manual_route_override_count = 1 THEN '1 override'
    ELSE '2+ overrides'
  END AS override_group,

  COUNT(d.delivery_id) AS deliveries,

  SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,

  ROUND(
    100.0 * SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END)
    / COUNT(d.delivery_id), 2
  ) AS failure_rate_pct,

  SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed,

  ROUND(
    100.0 * SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END)
    / COUNT(d.delivery_id), 2
  ) AS delay_rate_pct,

  ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_rating,

  ROUND(
    100.0 * SUM(CASE WHEN c.complaint_count > 0 THEN 1 ELSE 0 END)
    / COUNT(d.delivery_id), 2
  ) AS complaint_rate_pct

FROM deliveries d
JOIN orders o
  ON d.order_id = o.order_id

LEFT JOIN complaint_summary c
  ON o.order_id = c.order_id

GROUP BY override_group
ORDER BY failure_rate_pct DESC;
"

override_results <- dbGetQuery(conn, query_override)
print(override_results)

  override_group deliveries failed failure_rate_pct delayed delay_rate_pct
1     1 override        310     51            16.45      67          21.61
2   2+ overrides        241     35            14.52      57          23.65
3    No override        399     46            11.53      78          19.55
  avg_rating complaint_rate_pct
1       3.88              20.00
2       3.78              22.41
3       3.90              23.31


In [12]:
%%R

query_maintenance <- "
WITH complaint_summary AS (
  SELECT order_id, COUNT(complaint_id) AS complaint_count
  FROM complaints
  GROUP BY order_id
)

SELECT
  v.maintenance_status,

  COUNT(d.delivery_id) AS deliveries,

  SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,

  ROUND(
    100.0 * SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END)
    / COUNT(d.delivery_id), 2
  ) AS failure_rate_pct,

  SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed,

  ROUND(
    100.0 * SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END)
    / COUNT(d.delivery_id), 2
  ) AS delay_rate_pct,

  ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_rating,

  ROUND(
    100.0 * SUM(CASE WHEN c.complaint_count > 0 THEN 1 ELSE 0 END)
    / COUNT(d.delivery_id), 2
  ) AS complaint_rate_pct,

  ROUND(AVG(v.battery_health_pct), 2) AS avg_battery_pct

FROM deliveries d
JOIN vehicles v
  ON d.vehicle_id = v.vehicle_id

JOIN orders o
  ON d.order_id = o.order_id

LEFT JOIN complaint_summary c
  ON o.order_id = c.order_id

LEFT JOIN incidents i
  ON d.delivery_id = i.delivery_id

GROUP BY v.maintenance_status
ORDER BY failure_rate_pct DESC;
"

vehicle_results <- dbGetQuery(conn, query_maintenance)
print(vehicle_results)

  maintenance_status deliveries failed failure_rate_pct delayed delay_rate_pct
1           InRepair        268     81            30.22      54          20.15
2             Active        554     45             8.12     115          20.76
3          Scheduled        160     11             6.88      38          23.75
  avg_rating complaint_rate_pct avg_battery_pct
1       3.64              26.49           76.41
2       3.95              19.68           76.40
3       3.92              22.50           78.70
